# AI-Driven Threat Intelligence and Incident Response System
## Real-Time RAG for Cybersecurity

This notebook implements a comprehensive threat intelligence system using:
- **RAG (Retrieval-Augmented Generation)** for real-time threat intelligence
- **LLM-based threat detection** and classification
- **Automated incident response** recommendations
- **Real-time security log analysis**

### System Architecture:
1. Threat Intelligence Knowledge Base (Vector Store)
2. Security Log Ingestion & Parsing
3. RAG Pipeline for Threat Retrieval
4. LLM-based Threat Classification
5. Incident Response Engine
6. Real-time Monitoring Dashboard

## 1. Setup and Imports

In [ ]:
# Install required packages
!pip install -q langchain langchain-community langchain-openai chromadb sentence-transformers openai python-dotenv pandas numpy matplotlib seaborn plotly

In [ ]:
import os
import json
import datetime
import random
import time
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from collections import defaultdict

# Data processing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# LangChain components
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.schema import Document

# Environment
from dotenv import load_dotenv

# Set style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All imports successful")

In [ ]:
# Load environment variables
load_dotenv()

# Configure API key (using OpenAI, but you can substitute with other providers)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not OPENAI_API_KEY:
    print("⚠️  Warning: OPENAI_API_KEY not found in environment")
    print("Please set your API key:")
    print("Option 1: Create a .env file with OPENAI_API_KEY=your-key")
    print("Option 2: Set it directly: os.environ['OPENAI_API_KEY'] = 'your-key'")
else:
    print("✓ API key loaded successfully")

## 2. Threat Intelligence Knowledge Base

Creating a comprehensive threat intelligence database covering:
- Common attack patterns (MITRE ATT&CK framework)
- CVE vulnerabilities
- Threat actor profiles
- Incident response playbooks

In [ ]:
# Comprehensive threat intelligence data
THREAT_INTELLIGENCE_DATA = [
    {
        "id": "T1190",
        "category": "Initial Access",
        "technique": "Exploit Public-Facing Application",
        "description": "Adversaries may attempt to exploit weaknesses in Internet-facing applications to gain initial access. SQL injection, command injection, and cross-site scripting are common methods.",
        "indicators": ["abnormal HTTP request patterns", "SQL error messages", "unusual URL parameters", "multiple 403/401 errors followed by 200"],
        "severity": "CRITICAL",
        "response": "Immediately isolate affected systems. Review WAF logs. Patch vulnerable applications. Conduct forensic analysis of compromised data."
    },
    {
        "id": "T1110",
        "category": "Credential Access",
        "technique": "Brute Force Attack",
        "description": "Adversaries may use brute force techniques to gain access to accounts when passwords are unknown or when password hashes are obtained. Multiple failed login attempts from same source IP.",
        "indicators": ["multiple failed authentication attempts", "account lockouts", "authentication from unusual geolocations", "password spray patterns"],
        "severity": "HIGH",
        "response": "Enable account lockout policies. Implement MFA. Block source IP addresses. Review authentication logs for compromised accounts. Reset credentials for targeted accounts."
    },
    {
        "id": "T1566",
        "category": "Initial Access",
        "technique": "Phishing",
        "description": "Adversaries may send phishing messages to gain access to victim systems. Spearphishing with malicious attachments or links is common.",
        "indicators": ["suspicious email attachments", "URLs with domain typosquatting", "requests to credential harvesting sites", "macro-enabled documents"],
        "severity": "HIGH",
        "response": "Quarantine suspicious emails. Block malicious domains/IPs. Scan endpoints for IOCs. Conduct user awareness training. Review email gateway logs."
    },
    {
        "id": "T1486",
        "category": "Impact",
        "technique": "Ransomware",
        "description": "Adversaries may encrypt data on target systems to interrupt availability and demand ransom payment. Often involves file encryption and deletion of backups.",
        "indicators": ["rapid file modifications", "file extension changes", "ransom notes", "backup deletion", "encrypted file extensions"],
        "severity": "CRITICAL",
        "response": "Immediately isolate infected systems from network. DO NOT pay ransom. Restore from clean backups. Identify patient zero. Analyze encryption algorithm. Report to law enforcement."
    },
    {
        "id": "T1071",
        "category": "Command and Control",
        "technique": "Application Layer Protocol - C2 Communication",
        "description": "Adversaries may communicate using application layer protocols to avoid detection. HTTP/HTTPS traffic to known malicious domains or unusual DNS queries.",
        "indicators": ["beaconing to external IPs", "DNS tunneling", "unusual HTTPS traffic volumes", "connections to newly registered domains"],
        "severity": "CRITICAL",
        "response": "Block C2 domains/IPs immediately. Isolate affected systems. Capture network traffic for analysis. Hunt for additional compromised hosts. Review firewall and proxy logs."
    },
    {
        "id": "T1059",
        "category": "Execution",
        "technique": "Command and Scripting Interpreter",
        "description": "Adversaries may abuse command and script interpreters to execute commands. PowerShell, Python, Bash scripts used for malicious purposes.",
        "indicators": ["obfuscated PowerShell commands", "Base64 encoded scripts", "unusual script execution", "cmd.exe spawning from unusual processes"],
        "severity": "HIGH",
        "response": "Enable PowerShell logging and script block logging. Analyze command history. Terminate malicious processes. Implement application whitelisting. Review EDR alerts."
    },
    {
        "id": "T1078",
        "category": "Persistence",
        "technique": "Valid Accounts - Compromised Credentials",
        "description": "Adversaries may use stolen credentials to gain access and maintain persistence. Includes use of default credentials or credential dumping.",
        "indicators": ["impossible travel", "access from unusual locations", "privilege escalation", "access outside business hours"],
        "severity": "CRITICAL",
        "response": "Force password reset for compromised accounts. Review access logs. Enable MFA. Conduct full credential audit. Hunt for lateral movement. Review privileged account usage."
    },
    {
        "id": "T1003",
        "category": "Credential Access",
        "technique": "Credential Dumping",
        "description": "Adversaries may attempt to dump credentials to obtain account login information. LSASS memory dumping, SAM database extraction, or Kerberos ticket extraction.",
        "indicators": ["LSASS process access", "Mimikatz execution", "SAM/SYSTEM file access", "unusual process memory reads"],
        "severity": "CRITICAL",
        "response": "Immediately rotate all credentials. Enable Credential Guard. Hunt for lateral movement. Review domain controller logs. Analyze memory dumps. Check for Golden Ticket attacks."
    },
    {
        "id": "T1570",
        "category": "Lateral Movement",
        "technique": "Lateral Tool Transfer",
        "description": "Adversaries may transfer tools between systems to avoid detection and support remote execution. SMB file transfers, remote desktop, or custom protocols.",
        "indicators": ["unusual SMB traffic", "file transfers to multiple systems", "remote desktop sessions", "tool staging in temp directories"],
        "severity": "HIGH",
        "response": "Segment network immediately. Block lateral movement paths. Analyze file hashes. Hunt across environment. Review admin tool usage. Enable enhanced logging on critical systems."
    },
    {
        "id": "T1048",
        "category": "Exfiltration",
        "technique": "Data Exfiltration",
        "description": "Adversaries may exfiltrate data over alternative protocols to avoid detection. DNS exfiltration, ICMP tunneling, or encrypted channels.",
        "indicators": ["large outbound data transfers", "unusual DNS query volumes", "uploads to cloud storage", "encrypted traffic to unknown destinations"],
        "severity": "CRITICAL",
        "response": "Block exfiltration channels immediately. Capture network traffic. Identify stolen data. Review DLP policies. Analyze destination servers. Notify legal/compliance teams."
    },
    {
        "id": "CVE-2021-44228",
        "category": "Vulnerability",
        "technique": "Log4Shell - Remote Code Execution",
        "description": "Critical vulnerability in Apache Log4j allows remote code execution via JNDI lookup. Widely exploited in the wild.",
        "indicators": ["JNDI lookup strings in logs", "${jndi:ldap references", "unusual LDAP connections", "Java process spawning shells"],
        "severity": "CRITICAL",
        "response": "Immediately patch Log4j to version 2.17.0+. Apply WAF rules. Hunt for exploitation attempts. Review Java application logs. Check for webshells and persistence mechanisms."
    },
    {
        "id": "CVE-2023-23397",
        "category": "Vulnerability",
        "technique": "Microsoft Outlook Privilege Escalation",
        "description": "Critical Outlook vulnerability allowing NTLM hash theft via specially crafted calendar invites.",
        "indicators": ["unusual calendar invites", "SMB connections to external IPs", "NTLM authentication to unknown servers"],
        "severity": "CRITICAL",
        "response": "Apply Microsoft patches immediately. Block SMB at perimeter. Review authentication logs for hash relay attacks. Reset credentials if compromise suspected."
    },
    {
        "id": "APT-001",
        "category": "Threat Actor",
        "technique": "APT29 (Cozy Bear) - State-Sponsored",
        "description": "Russian state-sponsored APT group known for sophisticated supply chain attacks, spearphishing, and long-term persistence.",
        "indicators": ["Sunburst/Solorigate backdoor", "Cobalt Strike beacons", "living-off-the-land techniques", "cloud infrastructure abuse"],
        "severity": "CRITICAL",
        "response": "Engage incident response team immediately. Preserve forensic evidence. Hunt for advanced persistent threats. Review cloud service logs. Consider full environment rebuild."
    },
    {
        "id": "DDoS-001",
        "category": "Availability",
        "technique": "Distributed Denial of Service Attack",
        "description": "Volumetric attack designed to overwhelm network or application resources. Can be HTTP floods, SYN floods, or amplification attacks.",
        "indicators": ["traffic spike from multiple sources", "service degradation", "unusual traffic patterns", "amplification attack signatures"],
        "severity": "HIGH",
        "response": "Enable DDoS mitigation services. Implement rate limiting. Block malicious IPs. Use CDN/cloud scrubbing. Analyze attack vectors. Scale infrastructure if needed."
    },
    {
        "id": "INSIDER-001",
        "category": "Insider Threat",
        "technique": "Malicious Insider Data Theft",
        "description": "Authorized user misusing access privileges to exfiltrate sensitive data. Often occurs before employee departure.",
        "indicators": ["unusual data access patterns", "bulk downloads", "access to unrelated systems", "use of personal storage devices"],
        "severity": "HIGH",
        "response": "Review user activity logs. Disable account access. Conduct HR investigation. Analyze data accessed. Review DLP alerts. Preserve evidence for legal proceedings."
    }
]

print(f"✓ Loaded {len(THREAT_INTELLIGENCE_DATA)} threat intelligence entries")

## 3. Security Log Generation

Simulating realistic security log data for testing

In [ ]:
@dataclass
class SecurityLog:
    timestamp: str
    source_ip: str
    destination_ip: str
    event_type: str
    severity: str
    description: str
    user: Optional[str] = None
    raw_log: Optional[str] = None
    
class SecurityLogGenerator:
    """Generate realistic security logs for testing"""
    
    def __init__(self):
        self.event_types = [
            'failed_login', 'successful_login', 'sql_injection_attempt',
            'xss_attempt', 'port_scan', 'brute_force', 'malware_detected',
            'ransomware_activity', 'suspicious_dns_query', 'c2_communication',
            'data_exfiltration', 'privilege_escalation', 'lateral_movement',
            'credential_dumping', 'phishing_email', 'ddos_attack'
        ]
        
        self.internal_ips = [f"192.168.1.{i}" for i in range(1, 50)]
        self.external_ips = [
            "203.0.113.42", "198.51.100.23", "185.220.101.45",
            "45.142.212.61", "89.248.172.16", "23.129.64.131"
        ]
        self.users = ["admin", "jdoe", "asmith", "root", "service_account", "unknown"]
    
    def generate_log(self, event_type: Optional[str] = None) -> SecurityLog:
        """Generate a single security log entry"""
        if event_type is None:
            event_type = random.choice(self.event_types)
        
        timestamp = datetime.datetime.now() - datetime.timedelta(
            minutes=random.randint(0, 1440)
        )
        
        # Event-specific patterns
        if 'login' in event_type:
            source_ip = random.choice(self.external_ips + self.internal_ips)
            dest_ip = random.choice(self.internal_ips)
            user = random.choice(self.users)
            severity = 'LOW' if 'successful' in event_type else 'MEDIUM'
            desc = f"{'Successful' if 'successful' in event_type else 'Failed'} login attempt for user {user}"
        
        elif 'injection' in event_type or 'xss' in event_type:
            source_ip = random.choice(self.external_ips)
            dest_ip = random.choice(self.internal_ips)
            user = None
            severity = 'CRITICAL'
            attack_type = 'SQL injection' if 'sql' in event_type else 'XSS'
            desc = f"{attack_type} attempt detected from {source_ip}"
        
        elif 'brute_force' in event_type:
            source_ip = random.choice(self.external_ips)
            dest_ip = random.choice(self.internal_ips)
            user = random.choice(self.users)
            severity = 'HIGH'
            desc = f"Multiple failed login attempts ({random.randint(10, 100)}) for user {user} from {source_ip}"
        
        elif 'ransomware' in event_type:
            source_ip = random.choice(self.internal_ips)
            dest_ip = random.choice(self.internal_ips)
            user = random.choice(self.users)
            severity = 'CRITICAL'
            desc = f"Ransomware activity detected: rapid file encryption on {dest_ip}"
        
        elif 'c2_communication' in event_type:
            source_ip = random.choice(self.internal_ips)
            dest_ip = random.choice(self.external_ips)
            user = None
            severity = 'CRITICAL'
            desc = f"Suspected C2 beaconing detected from {source_ip} to {dest_ip}"
        
        elif 'exfiltration' in event_type:
            source_ip = random.choice(self.internal_ips)
            dest_ip = random.choice(self.external_ips)
            user = random.choice(self.users)
            severity = 'CRITICAL'
            desc = f"Large data transfer ({random.randint(100, 5000)}MB) to external IP {dest_ip}"
        
        else:
            source_ip = random.choice(self.external_ips + self.internal_ips)
            dest_ip = random.choice(self.internal_ips)
            user = random.choice(self.users) if random.random() > 0.5 else None
            severity = random.choice(['LOW', 'MEDIUM', 'HIGH'])
            desc = f"{event_type.replace('_', ' ').title()} detected"
        
        raw_log = f"{timestamp.isoformat()} {source_ip} -> {dest_ip} {event_type.upper()} {desc}"
        
        return SecurityLog(
            timestamp=timestamp.isoformat(),
            source_ip=source_ip,
            destination_ip=dest_ip,
            event_type=event_type,
            severity=severity,
            description=desc,
            user=user,
            raw_log=raw_log
        )
    
    def generate_logs(self, count: int = 100) -> List[SecurityLog]:
        """Generate multiple security logs"""
        return [self.generate_log() for _ in range(count)]

# Test the generator
log_generator = SecurityLogGenerator()
sample_logs = log_generator.generate_logs(50)
print(f"✓ Generated {len(sample_logs)} sample security logs")
print("\nSample log:")
print(f"  Timestamp: {sample_logs[0].timestamp}")
print(f"  Event: {sample_logs[0].event_type}")
print(f"  Severity: {sample_logs[0].severity}")
print(f"  Description: {sample_logs[0].description}")

## 4. Vector Store Setup - RAG Implementation

Building the vector database for threat intelligence retrieval

In [ ]:
class ThreatIntelligenceRAG:
    """RAG system for threat intelligence retrieval"""
    
    def __init__(self, threat_data: List[Dict], use_openai: bool = True):
        self.threat_data = threat_data
        self.use_openai = use_openai
        
        # Choose embedding model
        if use_openai and OPENAI_API_KEY:
            print("Using OpenAI embeddings...")
            self.embeddings = OpenAIEmbeddings()
        else:
            print("Using HuggingFace embeddings (free, local)...")
            self.embeddings = HuggingFaceEmbeddings(
                model_name="sentence-transformers/all-MiniLM-L6-v2"
            )
        
        self.vectorstore = None
        self.retriever = None
        
    def build_vectorstore(self):
        """Build vector store from threat intelligence data"""
        print("Building threat intelligence vector store...")
        
        # Create documents from threat data
        documents = []
        for threat in self.threat_data:
            content = f"""
ID: {threat['id']}
Category: {threat['category']}
Technique: {threat['technique']}
Severity: {threat['severity']}

Description:
{threat['description']}

Indicators of Compromise:
{', '.join(threat['indicators'])}

Incident Response:
{threat['response']}
"""
            metadata = {
                'id': threat['id'],
                'category': threat['category'],
                'technique': threat['technique'],
                'severity': threat['severity']
            }
            documents.append(Document(page_content=content, metadata=metadata))
        
        # Create vector store
        self.vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=self.embeddings,
            collection_name="threat_intelligence"
        )
        
        # Create retriever
        self.retriever = self.vectorstore.as_retriever(
            search_kwargs={"k": 3}  # Retrieve top 3 most relevant threats
        )
        
        print(f"✓ Vector store built with {len(documents)} threat intelligence documents")
    
    def retrieve_threats(self, query: str, k: int = 3) -> List[Document]:
        """Retrieve relevant threats based on query"""
        if not self.vectorstore:
            raise ValueError("Vector store not built. Call build_vectorstore() first.")
        
        results = self.vectorstore.similarity_search(query, k=k)
        return results

# Initialize RAG system
rag_system = ThreatIntelligenceRAG(THREAT_INTELLIGENCE_DATA, use_openai=False)
rag_system.build_vectorstore()

In [ ]:
# Test RAG retrieval
test_query = "Multiple failed login attempts from external IP address"
print(f"Test Query: {test_query}\n")

retrieved_threats = rag_system.retrieve_threats(test_query, k=3)
print(f"Retrieved {len(retrieved_threats)} relevant threats:\n")

for i, doc in enumerate(retrieved_threats, 1):
    print(f"{i}. {doc.metadata['technique']} ({doc.metadata['severity']})")
    print(f"   ID: {doc.metadata['id']}")
    print(f"   Category: {doc.metadata['category']}")
    print()

## 5. LLM-Based Threat Analysis Engine

In [ ]:
class ThreatAnalysisEngine:
    """LLM-based threat analysis with RAG integration"""
    
    def __init__(self, rag_system: ThreatIntelligenceRAG):
        self.rag_system = rag_system
        
        # Initialize LLM (use OpenAI or fallback to local model)
        if OPENAI_API_KEY:
            self.llm = ChatOpenAI(
                model_name="gpt-3.5-turbo",
                temperature=0,
                openai_api_key=OPENAI_API_KEY
            )
            self.use_llm = True
        else:
            print("⚠️  No LLM available. Using rule-based analysis.")
            self.llm = None
            self.use_llm = False
        
        self.prompt_template = PromptTemplate(
            input_variables=["log_data", "threat_intel"],
            template="""
You are a cybersecurity threat analyst. Analyze the following security log and provide:
1. Threat classification
2. Severity assessment (CRITICAL, HIGH, MEDIUM, LOW)
3. Potential impact
4. Recommended response actions

Security Log:
{log_data}

Relevant Threat Intelligence:
{threat_intel}

Provide a structured analysis with clear, actionable recommendations.
"""
        )
    
    def analyze_log(self, log: SecurityLog) -> Dict[str, Any]:
        """Analyze a security log using RAG and LLM"""
        
        # Retrieve relevant threat intelligence
        query = f"{log.event_type} {log.description}"
        retrieved_threats = self.rag_system.retrieve_threats(query, k=2)
        
        threat_intel_context = "\n\n".join([
            f"Threat ID: {doc.metadata['id']}\n{doc.page_content}"
            for doc in retrieved_threats
        ])
        
        log_data = f"""
Timestamp: {log.timestamp}
Event Type: {log.event_type}
Source IP: {log.source_ip}
Destination IP: {log.destination_ip}
User: {log.user or 'N/A'}
Initial Severity: {log.severity}
Description: {log.description}
"""
        
        if self.use_llm:
            # Use LLM for advanced analysis
            prompt = self.prompt_template.format(
                log_data=log_data,
                threat_intel=threat_intel_context
            )
            response = self.llm.invoke(prompt)
            analysis = response.content
        else:
            # Fallback to rule-based analysis
            analysis = self._rule_based_analysis(log, retrieved_threats)
        
        return {
            'log': log,
            'retrieved_threats': retrieved_threats,
            'analysis': analysis,
            'timestamp': datetime.datetime.now().isoformat()
        }
    
    def _rule_based_analysis(self, log: SecurityLog, threats: List[Document]) -> str:
        """Rule-based analysis when LLM is not available"""
        
        analysis = f"""THREAT ANALYSIS REPORT
====================

Event: {log.event_type.upper()}
Severity: {log.severity}
Time: {log.timestamp}

CLASSIFICATION:
This event matches the following threat patterns:
"""
        for i, threat in enumerate(threats, 1):
            analysis += f"\n{i}. {threat.metadata['technique']} ({threat.metadata['id']})"
            analysis += f"\n   Severity: {threat.metadata['severity']}"
        
        analysis += f"""

POTENTIAL IMPACT:
- Source: {log.source_ip}
- Target: {log.destination_ip}
- User Context: {log.user or 'Unknown'}

RECOMMENDED ACTIONS:
"""
        if threats:
            # Extract response from first matching threat
            for threat in threats:
                if 'response' in threat.page_content.lower():
                    lines = threat.page_content.split('\n')
                    for i, line in enumerate(lines):
                        if 'Incident Response:' in line and i + 1 < len(lines):
                            analysis += f"\n{lines[i+1]}"
                            break
                    break
        
        return analysis

# Initialize threat analysis engine
analysis_engine = ThreatAnalysisEngine(rag_system)
print("✓ Threat analysis engine initialized")

In [ ]:
# Test threat analysis
test_log = log_generator.generate_log('brute_force')
print("Analyzing security log...\n")

result = analysis_engine.analyze_log(test_log)

print("="*60)
print("SECURITY LOG ANALYSIS")
print("="*60)
print(f"\nOriginal Log:")
print(f"  {test_log.raw_log}")
print(f"\nMatched Threats:")
for threat in result['retrieved_threats']:
    print(f"  - {threat.metadata['technique']} ({threat.metadata['id']})")
print(f"\n{result['analysis']}")
print("="*60)

## 6. Real-Time Monitoring System

In [ ]:
class RealTimeSecurityMonitor:
    """Real-time security monitoring and incident response system"""
    
    def __init__(self, analysis_engine: ThreatAnalysisEngine, log_generator: SecurityLogGenerator):
        self.analysis_engine = analysis_engine
        self.log_generator = log_generator
        self.incident_history = []
        self.threat_statistics = defaultdict(int)
        
    def process_log_stream(self, duration_seconds: int = 30, logs_per_second: float = 2):
        """Simulate real-time log processing"""
        print(f"Starting real-time monitoring for {duration_seconds} seconds...\n")
        
        start_time = time.time()
        log_count = 0
        critical_incidents = []
        
        while (time.time() - start_time) < duration_seconds:
            # Generate new log
            log = self.log_generator.generate_log()
            log_count += 1
            
            # Analyze log
            result = self.analysis_engine.analyze_log(log)
            
            # Track statistics
            self.threat_statistics[log.event_type] += 1
            self.incident_history.append(result)
            
            # Alert on critical events
            if log.severity in ['CRITICAL', 'HIGH']:
                critical_incidents.append(result)
                print(f"🚨 [{log.severity}] {log.event_type}: {log.description}")
            
            # Throttle to simulate real-time
            time.sleep(1.0 / logs_per_second)
        
        print(f"\n✓ Processed {log_count} logs in {duration_seconds} seconds")
        print(f"✓ Critical incidents detected: {len(critical_incidents)}")
        
        return critical_incidents
    
    def get_statistics(self) -> pd.DataFrame:
        """Get monitoring statistics"""
        stats_df = pd.DataFrame([
            {'Event Type': k, 'Count': v}
            for k, v in self.threat_statistics.items()
        ]).sort_values('Count', ascending=False)
        
        return stats_df
    
    def generate_incident_report(self, critical_incidents: List[Dict]) -> str:
        """Generate incident response report"""
        report = f"""
SECURITY INCIDENT REPORT
Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
{'='*70}

SUMMARY:
Total Incidents Analyzed: {len(self.incident_history)}
Critical/High Severity: {len(critical_incidents)}

TOP THREATS DETECTED:
"""
        stats = self.get_statistics()
        for _, row in stats.head(5).iterrows():
            report += f"  - {row['Event Type']}: {row['Count']} occurrences\n"
        
        report += "\n" + "="*70 + "\n"
        report += "CRITICAL INCIDENTS REQUIRING IMMEDIATE ATTENTION:\n"
        report += "="*70 + "\n\n"
        
        for i, incident in enumerate(critical_incidents[:5], 1):
            log = incident['log']
            report += f"{i}. [{log.severity}] {log.event_type.upper()}\n"
            report += f"   Time: {log.timestamp}\n"
            report += f"   Source: {log.source_ip} → Destination: {log.destination_ip}\n"
            report += f"   Description: {log.description}\n"
            report += f"   Matched Threats: {', '.join([t.metadata['id'] for t in incident['retrieved_threats']])}\n"
            report += "\n"
        
        return report

# Initialize monitor
security_monitor = RealTimeSecurityMonitor(analysis_engine, log_generator)
print("✓ Real-time security monitor initialized")

In [ ]:
# Run real-time monitoring simulation
print("Starting real-time threat detection simulation...\n")
critical_incidents = security_monitor.process_log_stream(duration_seconds=20, logs_per_second=3)

## 7. Visualization Dashboard

In [ ]:
# Threat statistics
stats_df = security_monitor.get_statistics()
print("\nThreat Statistics:")
print(stats_df.to_string(index=False))

In [ ]:
# Visualization 1: Threat Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
stats_df.head(10).plot(kind='barh', x='Event Type', y='Count', ax=axes[0], color='crimson')
axes[0].set_title('Top 10 Security Events Detected', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Event Type')

# Pie chart for severity distribution
severity_counts = defaultdict(int)
for incident in security_monitor.incident_history:
    severity_counts[incident['log'].severity] += 1

axes[1].pie(
    severity_counts.values(),
    labels=severity_counts.keys(),
    autopct='%1.1f%%',
    colors=['#ff4444', '#ffaa44', '#ffdd44', '#44ff44'],
    startangle=90
)
axes[1].set_title('Severity Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Timeline of Critical Events
critical_events = [inc for inc in security_monitor.incident_history if inc['log'].severity in ['CRITICAL', 'HIGH']]

if critical_events:
    timeline_data = []
    for inc in critical_events:
        timeline_data.append({
            'timestamp': inc['log'].timestamp,
            'event_type': inc['log'].event_type,
            'severity': inc['log'].severity
        })
    
    timeline_df = pd.DataFrame(timeline_data)
    timeline_df['timestamp'] = pd.to_datetime(timeline_df['timestamp'])
    timeline_df = timeline_df.sort_values('timestamp')
    
    # Create interactive timeline with Plotly
    fig = go.Figure()
    
    severity_colors = {
        'CRITICAL': 'red',
        'HIGH': 'orange',
        'MEDIUM': 'yellow',
        'LOW': 'green'
    }
    
    for severity in ['CRITICAL', 'HIGH']:
        severity_data = timeline_df[timeline_df['severity'] == severity]
        if not severity_data.empty:
            fig.add_trace(go.Scatter(
                x=severity_data['timestamp'],
                y=severity_data['event_type'],
                mode='markers',
                name=severity,
                marker=dict(
                    size=12,
                    color=severity_colors[severity],
                    line=dict(width=2, color='white')
                )
            ))
    
    fig.update_layout(
        title='Critical Security Events Timeline',
        xaxis_title='Time',
        yaxis_title='Event Type',
        height=500,
        hovermode='closest'
    )
    
    fig.show()
else:
    print("No critical events to display")

In [ ]:
# Visualization 3: Source IP Analysis
source_ips = defaultdict(int)
for incident in security_monitor.incident_history:
    if incident['log'].severity in ['CRITICAL', 'HIGH']:
        source_ips[incident['log'].source_ip] += 1

if source_ips:
    ip_df = pd.DataFrame([
        {'IP Address': k, 'Threat Count': v}
        for k, v in sorted(source_ips.items(), key=lambda x: x[1], reverse=True)
    ]).head(10)
    
    fig = go.Figure(data=[
        go.Bar(
            x=ip_df['Threat Count'],
            y=ip_df['IP Address'],
            orientation='h',
            marker=dict(color='crimson')
        )
    ])
    
    fig.update_layout(
        title='Top Malicious Source IPs',
        xaxis_title='Number of Critical/High Threats',
        yaxis_title='Source IP',
        height=400
    )
    
    fig.show()
else:
    print("No source IP data to display")

## 8. Generate Incident Response Report

In [ ]:
# Generate comprehensive incident report
incident_report = security_monitor.generate_incident_report(critical_incidents)
print(incident_report)

# Save report to file
report_filename = f"incident_report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(report_filename, 'w') as f:
    f.write(incident_report)

print(f"\n✓ Report saved to: {report_filename}")

## 9. Interactive Threat Query System

In [ ]:
def query_threat_intelligence(query: str, top_k: int = 3):
    """
    Interactive function to query the threat intelligence database
    
    Example queries:
    - "How do I respond to ransomware?"
    - "What are indicators of credential dumping?"
    - "Tell me about SQL injection attacks"
    """
    print(f"Query: {query}\n")
    print("="*70)
    
    # Retrieve relevant threats
    results = rag_system.retrieve_threats(query, k=top_k)
    
    for i, doc in enumerate(results, 1):
        print(f"\nResult {i}: {doc.metadata['technique']}")
        print("-" * 70)
        print(doc.page_content)
    
    return results

# Example queries
print("Example Threat Intelligence Queries:\n")
query_threat_intelligence("How should I respond to a brute force attack?")

In [ ]:
# Another example query
query_threat_intelligence("What are the signs of data exfiltration?")

## 10. Advanced: Custom Threat Analysis

In [ ]:
def analyze_custom_log(event_type: str, source_ip: str, dest_ip: str, description: str):
    """
    Analyze a custom security log entry
    
    Usage:
    analyze_custom_log(
        event_type="suspicious_activity",
        source_ip="203.0.113.42",
        dest_ip="192.168.1.100",
        description="Unusual PowerShell execution with Base64 encoding detected"
    )
    """
    custom_log = SecurityLog(
        timestamp=datetime.datetime.now().isoformat(),
        source_ip=source_ip,
        destination_ip=dest_ip,
        event_type=event_type,
        severity="UNKNOWN",
        description=description,
        user="unknown",
        raw_log=f"{datetime.datetime.now().isoformat()} {source_ip} -> {dest_ip} {event_type} {description}"
    )
    
    result = analysis_engine.analyze_log(custom_log)
    
    print("="*70)
    print("CUSTOM LOG ANALYSIS")
    print("="*70)
    print(f"\nInput Log:")
    print(f"  Event: {event_type}")
    print(f"  Source: {source_ip} → Destination: {dest_ip}")
    print(f"  Description: {description}")
    print(f"\nMatched Threat Patterns:")
    for threat in result['retrieved_threats']:
        print(f"  - {threat.metadata['technique']} ({threat.metadata['id']}) - {threat.metadata['severity']}")
    print(f"\n{result['analysis']}")
    print("="*70)
    
    return result

# Example: Analyze a custom suspicious event
analyze_custom_log(
    event_type="suspicious_powershell",
    source_ip="192.168.1.50",
    dest_ip="192.168.1.10",
    description="PowerShell executed with -EncodedCommand and downloading from external IP"
)

## 11. System Summary and Recommendations

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  AI-DRIVEN THREAT INTELLIGENCE & INCIDENT RESPONSE SYSTEM            ║
║  Real-Time RAG Implementation for Cybersecurity                      ║
╚══════════════════════════════════════════════════════════════════════╝

SYSTEM CAPABILITIES:
✓ Real-time security log analysis
✓ RAG-based threat intelligence retrieval
✓ LLM-powered threat classification
✓ Automated incident response recommendations
✓ Interactive threat intelligence queries
✓ Visual security monitoring dashboard
✓ Comprehensive incident reporting

NEXT STEPS FOR PRODUCTION DEPLOYMENT:

1. DATA INTEGRATION:
   - Connect to real SIEM systems (Splunk, ELK, QRadar)
   - Integrate threat feeds (MISP, AlienVault OTX, Threat Connect)
   - Add CVE database integration
   - Connect to EDR/XDR platforms

2. MODEL FINE-TUNING:
   - Fine-tune LLM on organization-specific security logs
   - Train on historical incident data
   - Incorporate company-specific threat landscape
   - Implement continuous learning pipeline

3. AUTOMATION:
   - Implement SOAR integration (Phantom, Demisto)
   - Automate incident ticket creation
   - Deploy automated response playbooks
   - Enable automatic threat blocking

4. SCALING:
   - Deploy on cloud infrastructure
   - Implement distributed processing
   - Add real-time streaming (Kafka, Kinesis)
   - Scale vector database (Pinecone, Weaviate)

5. MONITORING:
   - Set up alerting (PagerDuty, Slack)
   - Create executive dashboards
   - Implement SLA tracking
   - Add performance metrics

6. SECURITY:
   - Encrypt sensitive data
   - Implement access controls
   - Add audit logging
   - Ensure compliance (SOC2, ISO 27001)

USAGE EXAMPLES:

# Query threat intelligence
query_threat_intelligence("How do I detect lateral movement?")

# Analyze custom log
analyze_custom_log(
    event_type="anomaly",
    source_ip="10.0.0.5",
    dest_ip="8.8.8.8",
    description="Unusual DNS queries to suspicious domain"
)

# Run real-time monitoring
critical_incidents = security_monitor.process_log_stream(
    duration_seconds=60,
    logs_per_second=5
)

╔══════════════════════════════════════════════════════════════════════╗
║  System Ready for Threat Detection and Incident Response            ║
╚══════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# Final statistics
print("\nFinal System Statistics:")
print(f"Total logs processed: {len(security_monitor.incident_history)}")
print(f"Critical incidents: {len([i for i in security_monitor.incident_history if i['log'].severity == 'CRITICAL'])}")
print(f"High severity incidents: {len([i for i in security_monitor.incident_history if i['log'].severity == 'HIGH'])}")
print(f"Unique threat types: {len(security_monitor.threat_statistics)}")
print(f"\nThreat intelligence database: {len(THREAT_INTELLIGENCE_DATA)} entries")
print(f"Vector store documents: {rag_system.vectorstore._collection.count() if rag_system.vectorstore else 0}")